In [15]:
import pandas as pd
from scipy.stats import mannwhitneyu
import numpy as np
import os
import networkx as nx
from py_modules.utils import calc_ANC_score
from matplotlib import pyplot as plt


import warnings
warnings.filterwarnings("ignore")


In [16]:


def get_graph_scores(df, graph_name, exps_to_exclude=[]):
    """Get ANC and pairwise connectivity scores for a specific graph."""
    graph_rows = df[df['G_label'] == graph_name]
    
    scores = {}
    for _, row in graph_rows.iterrows():
        if row['method_name'] in exps_to_exclude:
            continue
        method_name = row['method_name']
        scores[f"{method_name}_ANC"] = row['ANC']
        scores[f"{method_name}_pairwise_connectivity"] = row['end_pairwise_connectivity']
    
    return scores

def get_single_graph_rankings(scores):
    """Get rankings of models based on ANC and pairwise connectivity scores for a single graph."""
    anc_scores = {col: scores[col] for col in scores if col.endswith('_ANC')}
    pairwise_scores = {col: scores[col] for col in scores if col.endswith('_pairwise_connectivity')}
    
    anc_rankings = sorted(anc_scores.items(), key=lambda x: x[1])
    pairwise_rankings = sorted(pairwise_scores.items(), key=lambda x: x[1])
    
    # Extract model names from column names
    anc_rankings = [(col.rsplit('_', maxsplit=1)[0], score) for col, score in anc_rankings]
    pairwise_rankings = [(col.rsplit('_', maxsplit=2)[0], score) for col, score in pairwise_rankings]
    
    return anc_rankings, pairwise_rankings

def get_overall_rankings(df, exps_to_exclude=[]):
    """Get overall rankings of methods based on average rankings across graphs."""
    df = df[~df['method_name'].isin(exps_to_exclude)]
    # Get the list of unique graphs and methods
    graph_labels = df['G_label'].unique()
    method_names = df['method_name'].unique()


    
    # Initialize dictionaries to store rankings for each method
    anc_rankings_per_method = {method: [] for method in method_names}
    pairwise_rankings_per_method = {method: [] for method in method_names}
    
    # For each graph, rank the methods based on their scores
    for graph in graph_labels:
        # Filter the data for the current graph
        graph_df = df[df['G_label'] == graph]
        
        # Rank methods based on ANC (lower is better)
        graph_df['ANC_rank'] = graph_df['ANC'].rank(method='min', ascending=True)
        
        # Rank methods based on end_pairwise_connectivity (lower is better)
        graph_df['pairwise_rank'] = graph_df['end_pairwise_connectivity'].rank(method='min', ascending=True)
        
        # Append the rankings to the respective method in the dictionaries
        for method in method_names:
            method_rows = graph_df[graph_df['method_name'] == method]
            if not method_rows.empty:
                anc_rank = method_rows['ANC_rank'].values[0]
                pairwise_rank = method_rows['pairwise_rank'].values[0]
                anc_rankings_per_method[method].append(anc_rank)
                pairwise_rankings_per_method[method].append(pairwise_rank)
            else:
                # If a method has no score for this graph, we can choose to skip or assign the worst rank
                # Here, we'll assign the worst possible rank (number of methods + 1)
                worst_rank = len(method_names) + 1
                anc_rankings_per_method[method].append(worst_rank)
                pairwise_rankings_per_method[method].append(worst_rank)
    
    # Calculate the average rankings for each method
    avg_anc_rankings = []
    avg_pairwise_rankings = []
    for method in method_names:
        avg_anc_rank = sum(anc_rankings_per_method[method]) / len(anc_rankings_per_method[method])
        avg_pairwise_rank = sum(pairwise_rankings_per_method[method]) / len(pairwise_rankings_per_method[method])
        avg_anc_rankings.append((method, avg_anc_rank))
        avg_pairwise_rankings.append((method, avg_pairwise_rank))
    
    # Sort the methods based on average rankings
    anc_rankings_sorted = sorted(avg_anc_rankings, key=lambda x: x[1])
    pairwise_rankings_sorted = sorted(avg_pairwise_rankings, key=lambda x: x[1])
    
    return anc_rankings_sorted, pairwise_rankings_sorted

In [17]:
EXPS_INTEREST = ['MTSSL_MEGA_CrossAttention_1l_freeze_BA_FINDER_noProc', 
                 'MTSSL_MEGA_reset_BA_FINDER_noProc',
                 'ORIGINAL_BA_FINDER',
                 'HDA',
                 'degree',
                 'CI',
                 'random']# ['MTSSL_MEGA_freeze_BA_FINDER_noProc', 'ORIGINAL_BA_FINDER']
# results_df['method_name'].unique()


In [18]:
# Load the results
EXP = os.path.join('Evals', 'MEGA_integration_benchmark_4_p10')
# EXP = os.path.join('Evals', 'MUSE_CN_PPI')
# EXP = os.path.join('Evals', 'MEGA_integration_synthetic')
results_df_path = os.path.join(EXP, 'results.csv')
results_df = pd.read_csv(results_df_path)

# if method_name is in [MTSSL_MEGA_CrossAttention_1l_freeze_BA_FINDER_noProc,MTSSL_MEGA_freeze_BA_FINDER_noProc,MTSSL_MEGA_finetune_BA_FINDER_noProc], just do "best_of_three"
# methods_to_compare = [
#     'MTSSL_MEGA_CrossAttention_1l_freeze_BA_FINDER_noProc',
#     'MTSSL_MEGA_freeze_BA_FINDER_noProc',
#     'MTSSL_MEGA_finetune_BA_FINDER_noProc'
# ]

# # Create a filtered DataFrame with only the specified methods
# filtered_df = results_df[results_df['method_name'].isin(methods_to_compare)]

# # Group by G_label and find the row with minimum ANC score
# best_of_three = filtered_df.loc[filtered_df.groupby('G_label')['ANC'].idxmin()]

# # Change the method_Name to 'best_of_3'
# best_of_three['method_name'] = 'best_of_3'

# # Append the best_of_three rows to the original result_df
# results_df = pd.concat([results_df, best_of_three], ignore_index=True)

# # remove filtered_df from results_df
# results_df = results_df[~results_df['method_name'].isin(methods_to_compare)]

# print(results_df.shape)

# for duplicated G_label, method_name pairs, keep the one with the lowest ANC score
results_df = results_df.sort_values(by='ANC', ascending=True).drop_duplicates(subset=['G_label', 'method_name'])


exps_to_exclude = []# ['ORIGINAL_DiverseD_FINDER', 'MTSSL_MEGA_finetune_DiverseD_FINDER_proc']
for graph_name in results_df['G_label'].unique():
    G_path = os.path.join(EXP, f'{graph_name}', 'G.el')
    G = nx.read_edgelist(G_path, nodetype=int)
    n = G.number_of_nodes()
    print(f"{graph_name} has {n} nodes")
    results_df.loc[results_df['G_label'] == graph_name, 'n_nodes'] = n

# results_df = results_df[results_df['n_nodes'] > 500]


# sort results_df by G_label
results_df = results_df.sort_values(by='G_label')
if len(EXPS_INTEREST) > 0:
    results_df = results_df[results_df['method_name'].isin(EXPS_INTEREST)]

# Get scores for a specific graph
for graph_name in results_df['G_label'].unique():
    graph_scores = get_graph_scores(results_df, graph_name, exps_to_exclude)
    print()


    # Get rankings for a specific graph
    anc_rankings, pairwise_rankings = get_single_graph_rankings(graph_scores)
    print(f"\nRankings for {graph_name}:")
    print("ANC Rankings:")
    for rank, (model, score) in enumerate(anc_rankings, start=1):
        print(f"{rank}. {model}: {score}")
    print()
    print("\tPairwise Connectivity Rankings:")
    for rank, (model, score) in enumerate(pairwise_rankings, start=1):
        print(f"{rank}. {model}: {score}")

    print("--------------------------------")

# Get overall rankings
overall_anc_rankings, overall_pairwise_rankings = get_overall_rankings(results_df, exps_to_exclude)
print("\nOverall Rankings (ANC):")
for rank, (model, score) in enumerate(overall_anc_rankings, start=1):
    print(f"{rank}. {model}: {score}")
print("\nOverall Rankings (Pairwise Connectivity):")
for rank, (model, score) in enumerate(overall_pairwise_rankings, start=1):
    print(f"{rank}. {model}: {score}")



Bovine has 121 nodes
Ecoli has 328 nodes
yeast1 has 1966 nodes
Enron has 33696 nodes
humanDiseasome has 516 nodes
Treni_Roma has 255 nodes
powergrid has 4941 nodes
openflights has 1491 nodes
USAir97 has 332 nodes
OClinks has 1899 nodes
facebook has 4039 nodes
soc-wiki-Vote has 889 nodes
Circuit has 252 nodes
EU_flights has 1191 nodes
email-univ has 1133 nodes
Hamilton4000 has 4000 nodes
Hamilton3000b has 3000 nodes
Hamilton3000c has 3000 nodes
Hamilton2000 has 2000 nodes
Hamilton3000e has 3000 nodes
Hamilton3000a has 3000 nodes
Hamilton1000 has 1000 nodes
Hamilton3000d has 3000 nodes
Hamilton5000 has 5000 nodes
socfb-Caltech36 has 769 nodes
socfb-Reed98 has 962 nodes
socfb-Bowdoin47 has 2252 nodes
socfb-Haverford76 has 1446 nodes


Rankings for Bovine:
ANC Rankings:
1. HDA: 0.1142619375573921
2. degree: 0.1152662993572084
3. CI: 0.1180670339761248
4. ORIGINAL_BA_FINDER: 0.1188016528925619
5. MTSSL_MEGA_CrossAttention_1l_freeze_BA_FINDER_noProc: 0.1431932966023875
6. MTSSL_MEGA_reset_BA

In [19]:
results_df

,G_label,ANC,start_pairwise_connectivity,end_pairwise_connectivity,baseline,method_name,n_nodes
218,Bovine,0.114262,1.000000,0.000138,True,HDA,121.0
216,Bovine,0.898037,1.000000,0.795868,True,random,121.0
213,Bovine,0.143193,1.000000,0.000551,False,MTSSL_MEGA_CrossAttention_1l_freeze_BA_FINDER_...,121.0
212,Bovine,0.284016,1.000000,0.001515,False,MTSSL_MEGA_reset_BA_FINDER_noProc,121.0
211,Bovine,0.118802,1.000000,0.001653,False,ORIGINAL_BA_FINDER,121.0
...,...,...,...,...,...,...,...
129,yeast1,0.168156,0.701876,0.001752,True,degree,1966.0
128,yeast1,0.590316,0.701876,0.489161,True,random,1966.0
131,yeast1,0.232297,0.701876,0.014845,True,CI,1966.0
125,yeast1,0.203719,0.701876,0.001034,False,MTSSL_MEGA_CrossAttention_1l_freeze_BA_FINDER_...,1966.0


In [23]:

export_path = os.path.join(EXP, 'plots_limited')
os.makedirs(export_path, exist_ok=True)



EXPS_INTEREST = ['MTSSL_MEGA_CrossAttention_1l_freeze_BA_FINDER_noProc',
                #  'HDA',
                 'ORIGINAL_BA_FINDER',
                #  'MTSSL_MEGA_freeze_BA_FINDER_noProc',
                #  'MTSSL_MEGA_finetune_BA_FINDER_noProc',
                #  'MTSSL_MEGA_reset_BA_FINDER_noProc'
                 ] #results_df['method_name'].unique()


EXP_Labels = [
    'MUSE-CN',
    'FINDER'
]

COLORS = [
    'black',
    'gray'
]

STYLES = [
    '-',
    '--'
]


# set font size
plt.rcParams.update({'font.size': 20})

G_labels_interest = ['USAir97', 'facebook', 'Enron', 'openflights', 'Hamilton5000', 'powergrid', 'Hamilton4000']
# G_labels_interest = ['PPI_13']
# G_labels_interest = ['']
for graph_name in G_labels_interest:
    # if plot exists continue
    if os.path.exists(os.path.join(export_path, f"{graph_name}.png")):
        pass
    G_path = os.path.join(EXP, f'{graph_name}', 'G.el')
    G = nx.read_edgelist(G_path, nodetype=int)
    n = G.number_of_nodes()
    print(f"{graph_name} has {n} nodes")
    
    # create plot 
    fix, ax = plt.subplots(figsize=(10, 6))

    scores_list = []
    anc_scores_list = []
    for exp_i, exp in enumerate(EXPS_INTEREST):
        try:
            exp_sol_path = os.path.join(EXP, graph_name, f'{exp}_sol.txt')
            # read sol as list of ints
            sol = np.loadtxt(exp_sol_path, dtype=int)
            ANC_score, scores= calc_ANC_score(G, sol, conn_type='pairwise_connectivity', return_scores=True, normalize=True, step_size=1)
            scores_list.append(np.array(scores))

            anc_scores_list.append(ANC_score)
            ax.plot(scores, label=f"{EXP_Labels[exp_i]} (ANC: {100 * ANC_score:.2f}%)", color=COLORS[exp_i], linestyle=STYLES[exp_i], linewidth=3)
        except:
            print(f"No solution found for {exp} on {graph_name}")
            continue

    
    if len(scores_list) == 2:
        # (s_a - s_b)/ s_a
        # Calculate improvement percentage (assuming MUSE-CN is first and FINDER is second)
        improvement = ((anc_scores_list[1] - anc_scores_list[0]) / anc_scores_list[1] * 100).mean()
        print("For graph", graph_name, "the improvement is", improvement)
        # Fill between curves with blue dots
        ax.fill_between(range(len(scores_list[0])), 
                       scores_list[0], 
                       scores_list[1], 
                       color='green', 
                       alpha=0.2, 
                       hatch='...',
                       label=f'Improvement: {improvement:.1f}%')

    # title and labels

    ax.set_title(f"{graph_name} (|V|={n}, budget={int(n * 0.1)})")
    ax.set_xlabel("# of selected nodes")
    ax.set_ylabel("NPC")

    ax.set_ylim(0, 1)

    # y-ticks start from 0.2
    ax.set_yticks(np.arange(0.2, 1.1, 0.2))
    if graph_name == 'Enron':
        ax.legend(loc='upper right')
    else:
        ax.legend(loc='lower left')
    ax.set_xlim(0, None)
    # standardize x-ticks so depending on the scors lenghts, there are only 5 rounded ticks
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=5))
    # do'nt show just save 
    plt.savefig(os.path.join(export_path, f"{graph_name}.png"))
    plt.close()


# sort results_df by G_label
results_df = results_df.sort_values(by='G_label')

USAir97 has 332 nodes
For graph USAir97 the improvement is 24.255504494893156
facebook has 4039 nodes
For graph facebook the improvement is 24.472791464405695
Enron has 33696 nodes
For graph Enron the improvement is 22.375147463625648
openflights has 1491 nodes
For graph openflights the improvement is 31.151636091412616
Hamilton5000 has 5000 nodes
For graph Hamilton5000 the improvement is 4.16006856691296
powergrid has 4941 nodes
For graph powergrid the improvement is 5.85512136798325
Hamilton4000 has 4000 nodes
For graph Hamilton4000 the improvement is 4.096331406332434


In [24]:
np.mean([22.37, 4.16, 5.85, 24.47, 4.09])

12.187999999999999

# WandB output

In [9]:
EXPS_INTEREST = ['MTSSL_MEGA_freeze_BA_FINDER_noProc', 'ORIGINAL_BA_FINDER']


In [4]:
import wandb
import json

done_cache_file = "done_cache.json"
if os.path.exists(done_cache_file):
    with open(done_cache_file, "r") as f:
        done_cache = json.load(f)
else:
    done_cache = {}

# setup project
wandb.init(project="Q-CNDP-evaluation")

columns = ["graph_name", "n_nodes", "ANC_score", "connectivity_curve"]
results_table = wandb.Table(columns=columns)

for method in results_df['method_name'].unique():
    if method not in done_cache:
        done_cache[method] = []

    with wandb.init(project="Q-CNDP-evaluation", name=method, reinit=True) as run:
        method_results = []
        
        for graph_name in results_df['G_label'].unique():
            if graph_name in done_cache[method]:
                print(f"Skipping {method}-{graph_name}, already logged.")
                continue

            try:
                # Load graph
                G_path = os.path.join(EXP, f'{graph_name}', 'G.el')
                G = nx.read_edgelist(G_path, nodetype=int)
                n = G.number_of_nodes()
                
                # Load solution
                exp_sol_path = os.path.join(EXP, graph_name, f'{method}_sol.txt')
                sol = np.loadtxt(exp_sol_path, dtype=int)
                
                # Calculate scores
                ANC_score, scores = calc_ANC_score(
                    G, sol, 
                    conn_type='pairwise_connectivity',
                    return_scores=True, 
                    normalize=True, 
                    step_size=1
                )

                # Create a wandb.Table for line plotting:
                line_data = [[step, val] for step, val in enumerate(scores)]
                table = wandb.Table(data=line_data, columns=["step", "connectivity"])
                # Create a line plot from that table
                connectivity_plot = wandb.plot.line(
                    table,
                    x="step",
                    y="connectivity",
                    title=f"{graph_name}: Connectivity"
                )

                # Log the ANC score, raw node count, and line plot
                run.log({
                    f"ANC_score/{graph_name}": ANC_score,
                    f"n_nodes/{graph_name}": n,
                    f"scores/{graph_name}": connectivity_plot  # The line chart
                })

                # Optionally add an image version to a results_table
                fig, ax = plt.subplots(figsize=(8, 4))
                ax.plot(scores)
                ax.set_title(f"{graph_name} (n={n})")
                ax.set_xlabel("Step")
                ax.set_ylabel("Pairwise Connectivity")
                results_table.add_data(
                    graph_name,
                    n,
                    ANC_score,
                    wandb.Image(fig)
                )
                plt.close()

                method_results.append({
                    "graph_name": graph_name,
                    "ANC_score": ANC_score,
                    "n_nodes": n
                })

                done_cache[method].append(graph_name)

            except Exception as e:
                print(f"Error processing {graph_name} for {method}: {str(e)}")
                continue
        
        # Calculate and log summary metrics
        if method_results:
            avg_anc = np.mean([r["ANC_score"] for r in method_results])
            run.log({
                "average_ANC": avg_anc,
                "results_table": results_table
            })

# Save updated cache
with open(done_cache_file, "w") as f:
    json.dump(done_cache, f, indent=2)




wandb: WARNING Source type is set to 'repo' but some required information is missing from the environment. A job will not be created from this run. See https://docs.wandb.ai/guides/launch/create-job


wandb: WARNING Source type is set to 'repo' but some required information is missing from the environment. A job will not be created from this run. See https://docs.wandb.ai/guides/launch/create-job
